In [1]:
import cv2
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
from PIL import Image
from torchvision import datasets





In [2]:

data_dir = "../data/split/train"   # wherever your training data was
train_dataset = datasets.ImageFolder(data_dir)
classes = train_dataset.classes  # <-- list of class names in correct order
print(classes)
num_classes = len(classes)


['Akarna_Dhanurasana', 'Bharadvajas_Twist_pose_or_Bharadvajasana_I_', 'Boat_Pose_or_Paripurna_Navasana_', 'Bound_Angle_Pose_or_Baddha_Konasana_', 'Bow_Pose_or_Dhanurasana_', 'Bridge_Pose_or_Setu_Bandha_Sarvangasana_', 'Camel_Pose_or_Ustrasana_', 'Cat_Cow_Pose_or_Marjaryasana_', 'Chair_Pose_or_Utkatasana_', 'Child_Pose_or_Balasana_', 'Cobra_Pose_or_Bhujangasana_', 'Cockerel_Pose', 'Corpse_Pose_or_Savasana_', 'Cow_Face_Pose_or_Gomukhasana_', 'Crane_(Crow)_Pose_or_Bakasana_', 'Dolphin_Pose_or_Ardha_Pincha_Mayurasana_', 'Downward-Facing_Dog_pose_or_Adho_Mukha_Svanasana_', 'Eagle_Pose_or_Garudasana_', 'Eight-Angle_Pose_or_Astavakrasana_', 'Extended_Puppy_Pose_or_Uttana_Shishosana_', 'Extended_Revolved_Side_Angle_Pose_or_Utthita_Parsvakonasana_', 'Extended_Revolved_Triangle_Pose_or_Utthita_Trikonasana_', 'Feathered_Peacock_Pose_or_Pincha_Mayurasana_', 'Firefly_Pose_or_Tittibhasana_', 'Fish_Pose_or_Matsyasana_', 'Four-Limbed_Staff_Pose_or_Chaturanga_Dandasana_', 'Frog_Pose_or_Bhekasana', 'Gar

In [3]:



# Rebuild the model exactly as in training
model = models.resnet50(pretrained=False)  # no pretrained weights here
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, num_classes)
)

# Load your trained weights
state_dict = torch.load("torch_checkpoints/best_model.pth", map_location="cuda")
model.load_state_dict(state_dict)

# Put into evaluation mode
model.eval()



c:\Users\Harsh\.conda\envs\new_yoga\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Harsh\.conda\envs\new_yoga\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [4]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# transform = T.Compose([
#     T.Resize(320),
#     T.CenterCrop(288),
#     T.ToTensor(),
#     T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
# ])


transform = T.Compose([
    T.Resize(320),
    T.RandomResizedCrop(288, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.ToTensor(),
    T.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3), inplace=True),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
# gradients = None
# activations = None

# def save_gradient(module, grad_in, grad_out):
#     global gradients
#     gradients = grad_out[0]

# def save_activation(module, input, output):
#     global activations
#     activations = output

# # Hook last conv layer in ResNet50
# target_layer = model.layer4[2].conv3
# target_layer.register_forward_hook(save_activation)
# target_layer.register_backward_hook(save_gradient)

# def generate_gradcam(img_tensor, class_idx):
#     global gradients, activations
#     model.zero_grad()
#     output = model(img_tensor)
#     pred_class = output.argmax(dim=1).item()

#     # Backprop target class
#     class_score = output[0, class_idx]
#     class_score.backward()

#     # Global average pooling of gradients
#     pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])

#     # Weight activations
#     for i in range(activations.shape[1]):
#         activations[:, i, :, :] *= pooled_gradients[i]

#     # Average over channels
#     heatmap = torch.mean(activations, dim=1).squeeze().detach().cpu().numpy()
#     heatmap = np.maximum(heatmap, 0)
#     heatmap /= heatmap.max() if heatmap.max() != 0 else 1
#     return heatmap, pred_class

# def apply_gradcam(frame, heatmap):
#     heatmap = cv2.resize(heatmap, (frame.shape[1], frame.shape[0]))
#     heatmap = np.uint8(255 * heatmap)
#     heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
#     return cv2.addWeighted(frame, 0.6, heatmap, 0.4, 0)


In [ ]:
# cap = cv2.VideoCapture(0)  # 0 = default camera
# last_pred_time = 0
# prediction_interval = 5  # seconds

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # Show live feed
#     cv2.imshow("Yoga Pose Detector", frame)

#     # Run classification every few seconds
#     current_time = time.time()
#     if current_time - last_pred_time > prediction_interval:
#         last_pred_time = current_time

#         # Preprocess frame
#         img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         img_pil = Image.fromarray(img)
#         img_tensor = transform(img_pil).unsqueeze(0)

#         with torch.no_grad():
#             outputs = model(img_tensor)
#             _, predicted = outputs.max(1)
#             pose = classes[predicted.item()]
#             print(f"Detected Pose: {pose}")

#         # Overlay prediction on frame
#         cv2.putText(frame, pose, (20, 50), cv2.FONT_HERSHEY_SIMPLEX,
#                     1, (0, 255, 0), 2, cv2.LINE_AA)

#     # Show result with overlay
#     cv2.imshow("Yoga Pose Detector", frame)

#     # Press 'q' to quit
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()


Detected Pose: Seated_Forward_Bend_pose_or_Paschimottanasana_
Detected Pose: Virasana_or_Vajrasana
Detected Pose: Tree_Pose_or_Vrksasana_
Detected Pose: Tree_Pose_or_Vrksasana_
Detected Pose: Warrior_I_Pose_or_Virabhadrasana_I_
Detected Pose: Cat_Cow_Pose_or_Marjaryasana_
Detected Pose: Standing_Forward_Bend_pose_or_Uttanasana_
Detected Pose: Standing_Forward_Bend_pose_or_Uttanasana_
Detected Pose: Virasana_or_Vajrasana
Detected Pose: Wide-Angle_Seated_Forward_Bend_pose_or_Upavistha_Konasana_
Detected Pose: Yogic_sleep_pose
Detected Pose: Eagle_Pose_or_Garudasana_
Detected Pose: Yogic_sleep_pose


In [ ]:
# cap = cv2.VideoCapture(0)  # webcam
# prediction_interval = 2    # seconds
# last_pred_time = 0
# pose_name = "..."

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # Process every few seconds
#     current_time = cv2.getTickCount() / cv2.getTickFrequency()
#     if current_time - last_pred_time > prediction_interval:
#         last_pred_time = current_time

#         # Preprocess
#         img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         img_pil = Image.fromarray(img_rgb)
#         img_tensor = transform(img_pil).unsqueeze(0)

#         # Forward + Grad-CAM
#         with torch.no_grad():
#             outputs = model(img_tensor)
#             _, predicted = outputs.max(1)
        
#         heatmap, pred_class = generate_gradcam(img_tensor, class_idx=predicted.item())
#         pose_name = classes[pred_class]

#         # Overlay heatmap
#         frame = apply_gradcam(frame, heatmap)

#     # Draw pose name
#     cv2.putText(frame, f"Pose: {pose_name}", (20, 50),
#                 cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

#     cv2.imshow("Yoga Pose Detector (Grad-CAM)", frame)

#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()


c:\Users\Harsh\.conda\envs\new_yoga\Lib\site-packages\torch\nn\modules\module.py:1864: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


In [5]:
gradients = None
activations = None

def save_gradient(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0]

def save_activation(module, input, output):
    global activations
    activations = output

# Hook last conv layer
target_layer = model.layer4[2].conv3
target_layer.register_forward_hook(save_activation)
target_layer.register_backward_hook(save_gradient)

def generate_gradcam(img_tensor, class_idx):
    global gradients, activations
    model.zero_grad()
    output = model(img_tensor)

    class_score = output[0, class_idx]
    class_score.backward()

    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])

    for i in range(activations.shape[1]):
        activations[:, i, :, :] *= pooled_gradients[i]

    heatmap = torch.mean(activations, dim=1).squeeze().detach().cpu().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap /= heatmap.max() if heatmap.max() != 0 else 1
    return heatmap


In [6]:
cap = cv2.VideoCapture(0)
prediction_interval = 5
last_pred_time = 0
pose_name = "..."

while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_time = cv2.getTickCount() / cv2.getTickFrequency()
    if current_time - last_pred_time > prediction_interval:
        last_pred_time = current_time

        # Preprocess
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)
        img_tensor = transform(img_pil).unsqueeze(0)

        # Prediction
        outputs = model(img_tensor)
        _, predicted = outputs.max(1)
        pose_name = classes[predicted.item()]

        # Grad-CAM heatmap
        heatmap = generate_gradcam(img_tensor, class_idx=predicted.item())
        overlay = cv2.cvtColor(frame.copy(), cv2.COLOR_BGR2RGB)
        heatmap_resized = cv2.resize(heatmap, (overlay.shape[1], overlay.shape[0]))
        heatmap_resized = np.uint8(255 * heatmap_resized)
        heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
        overlay = cv2.addWeighted(frame, 0.6, heatmap_color, 0.4, 0)

        # Show Grad-CAM in separate window once per prediction
        cv2.imshow("Grad-CAM Snapshot", overlay)
        print(f"Predicted Pose: {pose_name}")

    # Live feed (pose name only)
    cv2.putText(frame, f"Pose: {pose_name}", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.imshow("Yoga Pose Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


c:\Users\Harsh\.conda\envs\new_yoga\Lib\site-packages\torch\nn\modules\module.py:1864: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Predicted Pose: Cow_Face_Pose_or_Gomukhasana_
Predicted Pose: Seated_Forward_Bend_pose_or_Paschimottanasana_
Predicted Pose: Half_Lord_of_the_Fishes_Pose_or_Ardha_Matsyendrasana_
Predicted Pose: Garland_Pose_or_Malasana_
Predicted Pose: Staff_Pose_or_Dandasana_
Predicted Pose: Staff_Pose_or_Dandasana_
Predicted Pose: Garland_Pose_or_Malasana_
Predicted Pose: Yogic_sleep_pose
Predicted Pose: Half_Lord_of_the_Fishes_Pose_or_Ardha_Matsyendrasana_
Predicted Pose: Cobra_Pose_or_Bhujangasana_
Predicted Pose: Cobra_Pose_or_Bhujangasana_


### learning summary of each class 
